# CS2309 — SwiftEdit Giai đoạn 3

Thực nghiệm cơ bản: **3a** ablation hyperparameter, **3c** batch metrics (runtime + ảnh).

### Setup (giống `CS2309_SwiftEdit_test.ipynb`)

| Môi trường | Thứ tự |
|---|---|
| **Colab T4** | Cell **1** clone + GPU check → Cell **2** pip/weights/HF |
| **macOS** | Cell 1 (nhận repo local) → Cell 2 `setup_macos.sh` |

Cell **1** = clone chuẩn từ test notebook: `nvidia-smi` → `git clone` → `/content/CS2309.CH201`. **Không** chỉ upload `.ipynb` lẻ.

Chỉnh `REPO_SLUG`, `USE_PRIVATE_REPO` ở cell 1 nếu fork / repo private (`GITHUB_TOKEN` trong Colab Secrets).

Kết quả: `results/phase3/ablation/`, `results/piebench/metrics.csv` (+ `edited_images/`)

**PIE-Bench:** tải qua [Google Form](https://forms.gle/hVMkTABb4uvZVjme9) (không có trên git) → cell **3b**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Tự nhận Colab (hoặc gán tay IN_COLAB = True/False)
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


_COLAB_GPU_ERR = (
    "Colab chưa có GPU.\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Colab extension (VS Code/Cursor): Select Kernel → Colab → New Colab Server "
    "→ Hardware accelerator: GPU → T4, rồi Restart kernel\n"
    "• Đang nối server CPU: Colab: Remove Server, tạo server GPU mới (không đổi GPU trên cùng server)"
)


def _check_colab_gpu() -> None:
    """Kiểm tra GPU sớm (trước clone/pip) — không cần torch."""
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK (nvidia-smi):", ", ".join(names))


# --- Colab: chỉ clone repo đề tài → lưu trên /content (không mount Drive) ---
REPO_SLUG = "NguyenKz/CS2309.CH201"  # đổi nếu fork: "user/CS2309.CH201"
USE_PRIVATE_REPO = True  # True chỉ khi repo private + đã thêm secret GITHUB_TOKEN trên Colab UI


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        print(
            "Không lấy được GITHUB_TOKEN (secret Colab). "
            "Dùng repo public hoặc: Colab → 🔑 Secrets → thêm GITHUB_TOKEN, rồi chạy lại.\n"
            f"Chi tiết: {e}\n"
            f"Fallback clone public: {public_url}"
        )
        return public_url


REPO_URL = _colab_repo_url() if IN_COLAB else f"https://github.com/{REPO_SLUG}.git"
COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if IN_COLAB:
    _check_colab_gpu()
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not ((PROJECT_ROOT / "SwiftEdit" / "infer.py").exists()):
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / "SwiftEdit" / "infer.py").exists():
                PROJECT_ROOT = p
                break

SWIFTEDIT_DIR = PROJECT_ROOT / "SwiftEdit"
WEIGHTS_DIR = SWIFTEDIT_DIR / "swiftedit_weights"
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(SWIFTEDIT_DIR)
if str(SWIFTEDIT_DIR) not in sys.path:
    sys.path.insert(0, str(SWIFTEDIT_DIR))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SWIFTEDIT_DIR:", SWIFTEDIT_DIR)
print("infer.py:", (SWIFTEDIT_DIR / "infer.py").is_file())
print("requirements.txt:", (SWIFTEDIT_DIR / "requirements.txt").is_file())
print("HF_HOME:", os.environ.get("HF_HOME", "(default ~/.cache)"))
print("Weights OK:", (WEIGHTS_DIR / "inverse_ckpt-120k").is_dir())
if IN_COLAB and not (SWIFTEDIT_DIR / "infer.py").is_file():
    raise FileNotFoundError(
        "Clone xong nhưng thiếu SwiftEdit/ — đảm bảo đã push SwiftEdit lên GitHub (không chỉ notebook)."
    )

### ② Setup — pip, weights, HF

Chạy sau cell 1 (đã clone + `PROJECT_ROOT` trên Colab).

In [ ]:
env = os.environ.copy()
env["REPO_SLUG"] = REPO_SLUG
env["COLAB_REPO_DIR"] = str(COLAB_REPO_DIR)
if IN_COLAB:
    env.setdefault("HF_HOME", "/content/huggingface")
    env.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

if IN_COLAB:
    if not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
        raise FileNotFoundError(
            f"Chưa có {PROJECT_ROOT}/SwiftEdit — chạy cell 1 (clone) trước."
        )
    setup_sh = PROJECT_ROOT / "scripts" / "setup_colab.sh"
    print("Chạy:", setup_sh)
    subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)
else:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_macos.sh"
    print("Chạy:", setup_sh)
    subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from load_setup_env import load_setup_env

PATHS = load_setup_env(PROJECT_ROOT)
for k, v in PATHS.items():
    print(f"{k}: {v}")

### Load models

In [ ]:
import time

import torch
from torchvision.utils import save_image

from infer import SWIFTEDIT_WEIGHTS_ROOT, edit_image, get_device
from models import AuxiliaryModel, IPSBV2Model, InverseModel

device = get_device()
print("device:", device)

inverse_ckpt = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "inverse_ckpt-120k")
path_unet_sb = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "sbv2_0.5")
ip_ckpt = os.path.join(SWIFTEDIT_WEIGHTS_ROOT, "ip_adapter_ckpt-90k/ip_adapter.bin")

t0 = time.time()
inverse_model = InverseModel(inverse_ckpt, device=device)
aux_model = AuxiliaryModel(device=device)
ip_sb_model = IPSBV2Model(
    path_unet_sb, ip_ckpt, aux_model, device=device, with_ip_mask_controller=True
)
print(f"Models ready in {time.time() - t0:.1f}s")

## 3a — Ablation hyperparameter

Thử `scale_edit` trên preset **dog** (`02.jpg`). Đổi `SCALE_TA`, `SCALE_NON_EDIT` nếu cần.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

ABLATION_DIR = Path(PATHS["ABLATION_DIR"])
IMG_PATH = str(Path(PATHS["SWIFTEDIT_DIR"]) / "assets/imgs_demo/02.jpg")
SRC_P = "dog"
EDIT_P = "dog with mouth opened"

SCALE_TA = 1.0
SCALE_NON_EDIT = 1.0
SCALE_EDIT_VALUES = [0.5, 1.0, 1.5]

input_img = Image.open(IMG_PATH).convert("RGB").resize((512, 512))
outputs = []
labels = []

for scale_edit in SCALE_EDIT_VALUES:
    t0 = time.time()
    result = edit_image(
        IMG_PATH,
        SRC_P,
        EDIT_P,
        inverse_model,
        aux_model,
        ip_sb_model,
        scale_ta=SCALE_TA,
        scale_edit=scale_edit,
        scale_non_edit=SCALE_NON_EDIT,
    )
    elapsed = time.time() - t0
    out_path = ABLATION_DIR / f"ablation_scale_edit_{scale_edit}.png"
    save_image(result, out_path)
    outputs.append(result.squeeze(0).permute(1, 2, 0).cpu().numpy())
    labels.append(f"s_edit={scale_edit}\n{elapsed:.1f}s")
    print(f"scale_edit={scale_edit} → {out_path} ({elapsed:.1f}s)")

fig, axes = plt.subplots(1, len(outputs) + 1, figsize=(4 * (len(outputs) + 1), 4))
axes[0].imshow(input_img)
axes[0].set_title("Input")
axes[0].axis("off")
for ax, out, lab in zip(axes[1:], outputs, labels):
    ax.imshow(out.clip(0, 1))
    ax.set_title(lab)
    ax.axis("off")
plt.tight_layout()
grid_path = ABLATION_DIR / "grid_scale_edit.png"
plt.savefig(grid_path, dpi=120)
plt.show()
print("Grid:", grid_path)

## 3b — Tải PIE-Bench

Dataset chính thức: [PnP Inversion / PIE-Bench](https://github.com/cure-lab/PnPInversion) — **700 mẫu**, có `mapping_file.json` + GT mask.

**Đầy đủ (700 mẫu):** form https://forms.gle/hVMkTABb4uvZVjme9 → zip → `PIEBENCH_ZIP`

**Test nhanh (2 ảnh demo, không cần form):** `python scripts/create_piebench_smoke.py` rồi eval với `--piebench-dir data/PIE-Bench-smoke`

In [ ]:
import subprocess

PIEBENCH_ZIP = ""  # ví dụ Colab: "/content/PIE-Bench.zip" sau khi upload
PIEBENCH_DIR = Path(PATHS.get("PIEBENCH_DIR") or PROJECT_ROOT / "data" / "PIE-Bench")

env = os.environ.copy()
env["PROJECT_ROOT"] = str(PROJECT_ROOT)
env["PIEBENCH_DIR"] = str(PIEBENCH_DIR)
if PIEBENCH_ZIP:
    env["PIEBENCH_ZIP"] = PIEBENCH_ZIP
    cmd = ["bash", str(PROJECT_ROOT / "scripts" / "download_piebench.sh"), PIEBENCH_ZIP]
else:
    cmd = ["bash", str(PROJECT_ROOT / "scripts" / "download_piebench.sh")]

print("Chạy:", " ".join(cmd))
r = subprocess.run(cmd, env=env, cwd=PROJECT_ROOT)
if r.returncode != 0:
    print(
        "\nChưa có data — tải form rồi set PIEBENCH_ZIP hoặc giải nén thủ công vào:",
        PIEBENCH_DIR,
    )
else:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    from piebench_utils import resolve_piebench_dir

    PB_ROOT = resolve_piebench_dir(PROJECT_ROOT, PIEBENCH_DIR)
    n_img = len(list((PB_ROOT / "annotation_images").rglob("*.jpg")))
    print("PIEBENCH_ROOT:", PB_ROOT)
    print("Số ảnh .jpg:", n_img)

## 3c — Demo nhanh (`imgs_demo`)

Preset demo trong repo — kiểm tra pipeline trước khi chạy PIE-Bench (cell 3d).

In [ ]:
import csv
from datetime import datetime, timezone

METRICS_DIR = Path(PATHS["METRICS_DIR"])
DEMO_DIR = Path(PATHS["SWIFTEDIT_DIR"]) / "assets" / "imgs_demo"

JOBS = [
    {"id": "dog", "img": "02.jpg", "src_p": "dog", "edit_p": "dog with mouth opened"},
    {
        "id": "woman",
        "img": "woman_face.jpg",
        "src_p": "woman",
        "edit_p": "Taylor Swift",
    },
]

rows = []
for job in JOBS:
    img_path = DEMO_DIR / job["img"]
    if not img_path.is_file():
        print("skip", job["id"], "— thiếu", img_path)
        continue
    t0 = time.time()
    result = edit_image(
        str(img_path),
        job["src_p"],
        job["edit_p"],
        inverse_model,
        aux_model,
        ip_sb_model,
    )
    elapsed = time.time() - t0
    safe = job["id"]
    out_img = METRICS_DIR / f"{safe}_output.png"
    save_image(result, out_img)
    rows.append(
        {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "job_id": job["id"],
            "device": device,
            "src_p": job["src_p"],
            "edit_p": job["edit_p"],
            "runtime_s": round(elapsed, 3),
            "output_path": str(out_img),
        }
    )
    print(f"{job['id']}: {job['src_p']!r} → {job['edit_p']!r} in {elapsed:.1f}s")

csv_path = METRICS_DIR / "metrics.csv"
fieldnames = list(rows[0].keys()) if rows else [
    "timestamp", "job_id", "device", "src_p", "edit_p", "runtime_s", "output_path"
]
write_header = not csv_path.exists()
with csv_path.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    if write_header:
        w.writeheader()
    w.writerows(rows)

print("Appended:", csv_path, f"({len(rows)} rows)")

In [ ]:
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from run_piebench_eval import run_piebench_eval

# --- cấu hình eval ---
MAX_SAMPLES = 10 if not IN_COLAB else 50  # Colab: 50; Mac thử 5–15
EDIT_CATEGORIES = None  # None = mọi loại; hoặc ["0","1",...]
RESUME = True

piebench_csv = run_piebench_eval(
    project_root=Path(PATHS["PROJECT_ROOT"]),
    inverse_model=inverse_model,
    aux_model=aux_model,
    ip_sb_model=ip_sb_model,
    edit_image_fn=edit_image,
    device=str(device),
    piebench_dir=PATHS.get("PIEBENCH_DIR") or None,
    output_dir=Path(PATHS.get("PIEBENCH_RESULTS_DIR") or PATHS["PROJECT_ROOT"] / "results" / "piebench"),
    max_samples=MAX_SAMPLES,
    edit_categories=EDIT_CATEGORIES,
    resume=RESUME,
)

df = pd.read_csv(piebench_csv)
display_cols = [
    "psnr_unedit", "mse_unedit", "clip_whole", "clip_edited", "runtime_s"
]
print("Rows:", len(df))
if len(df):
    print("\nMean metrics (batch vừa chạy):")
    print(df[display_cols].apply(pd.to_numeric, errors="coerce").mean(numeric_only=True))
    print("\nLưu tại:", piebench_csv)